# Preparando los Datos y Aplicando Transfer Learning

### De un dataset de texto a un modelo listo para entrenar

En el notebook anterior dejamos el modelo BERT cargado (con una cabeza de clasificacion que todavia no sabe nada) y el dataset de tweets de aerolineas limpio y guardado.  
Hoy vamos a terminar de preparar todo lo necesario para entrenar: convertir el texto en tensores que el modelo pueda procesar, y aplicar el paso central del transfer learning: congelar las capas base de BERT.

### Texto negativo vs. positivo

El dataset original tiene tres categorias de sentimiento: positive, neutral y negative. Para este ejercicio vamos a simplificar el problema a dos categorias (Positivo y Negativo), dejando de lado los tweets neutrales. Para que sea mas rapido de entrenar

### El plan de hoy

1. Retomar el dataset y quedarnos solo con tweets positivos y negativos.
2. Separar en conjunto de entrenamiento (train) y de prueba (test).
3. Tokenizar todo el dataset en batch y dejarlo en el formato que espera el modelo.
4. Cargar un modelo nuevo, ahora con dos categorias de salida.
5. Congelar las capas base del modelo (Fase 3 del temario) y verificar que el congelado funciono.
6. Bonus: repetir la preparacion con las tres categorias originales, para comparar mas adelante.

## 0. Preparando el entorno

Como esta puede ser una sesion nueva de Colab, volvemos a instalar las librerias e importar todo lo que vamos a necesitar. Si ya las tenes instaladas de la clase anterior, esta celda no hace nada nuevo.

In [ ]:
!pip install transformers datasets accelerate scikit-learn

In [ ]:
import re

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## 1. Retomando el dataset

Volvemos a cargar el dataset de tweets de aerolineas y aplicamos la misma limpieza liviana que usamos la clase pasada (sacar menciones y links). Lo generamos de nuevo desde la fuente para no depender de haber guardado un archivo en esta sesion de Colab.

In [ ]:
url_dataset = "https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv"

columnas_utiles = ["tweet_id", "airline", "text", "airline_sentiment"]
df = pd.read_csv(url_dataset, usecols=columnas_utiles)
df = df.rename(columns={
    "tweet_id": "id",
    "airline": "aerolinea",
    "text": "texto",
    "airline_sentiment": "sentimiento_original",
})


def quitar_menciones_y_links(texto):
    texto = re.sub(r"@\w+", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()


df["texto_sentimiento"] = df["texto"].apply(quitar_menciones_y_links)
df["sentimiento_original"].value_counts()

,count
sentimiento_original,
negative,9178
neutral,3099
positive,2363


## 2. Simplificando a un problema binario

Nos quedamos solo con los tweets positive y negative, y convertimos la etiqueta de texto a un numero: 0 para negative y 1 para positive. El modelo no entiende palabras como etiquetas, necesita numeros.

In [ ]:
etiqueta2id = {"negative": 0, "positive": 1}

# Nos quedamos solamente con los tweets que tienen una de estas dos etiquetas
df_binario = df[df["sentimiento_original"].isin(etiqueta2id.keys())].copy()

# Convertimos cada etiqueta de texto a su correspondiente ID numerico
df_binario["label"] = df_binario["sentimiento_original"].map(etiqueta2id)

df_binario["label"].value_counts()

,count
label,
0,9178
1,2363


## 3. Separando en train y test

Como siempre, necesitamos dejar una parte de los datos afuera del entrenamiento para poder evaluar el modelo con ejemplos que nunca vio. Usamos `stratify` para que la proporcion de tweets positivos y negativos sea parecida en ambos conjuntos.

Para que el entrenamiento en clase no tarde demasiado, tambien nos quedamos con una muestra mas chica del dataset. En tu tiempo libre podes repetir el ejercicio con el dataset completo.

In [ ]:
df_muestra = df_binario.sample(n=3000, random_state=42)

train_df, test_df = train_test_split(
    df_muestra,
    test_size=0.2,
    stratify=df_muestra["label"],
    random_state=42,
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 2400
Test: 600


## 4. Convirtiendo a un objeto Dataset

La libreria `datasets` de Hugging Face trabaja con un objeto propio llamado `Dataset`, pensado para manejar grandes volumenes de texto de forma eficiente y para conectarse directamente con el `Trainer` que vamos a usar en el proximo notebook. Lo armamos a partir de nuestro DataFrame de pandas, quedandonos solo con las columnas que necesitamos.

In [ ]:
# Convertimos el DataFrame de entrenamiento de pandas en un Dataset de Hugging Face
train_dataset = Dataset.from_pandas(
    train_df[["texto_sentimiento", "label"]].reset_index(drop=True) # quedamos solamente con las columnas que necesitamos
)
test_dataset = Dataset.from_pandas(
    test_df[["texto_sentimiento", "label"]].reset_index(drop=True)
)

train_dataset

Dataset({
    features: ['texto_sentimiento', 'label'],
    num_rows: 2400
})

## 5. Tokenizando el dataset completo

Necesitamos tokenizar los miles de tweets del dataset. Para eso escribimos una funcion que tokeniza un batch de textos a la vez, y se la pasamos a `.map()`.

Dos parametros importantes:
- `truncation=True`: si un tweet fuera mas largo que `max_length`, lo corta.
- `padding="max_length"`: si un tweet es mas corto, lo completa con tokens de relleno hasta llegar a `max_length`. Esto es necesario porque el modelo espera que todos los ejemplos de un batch tengan el mismo largo.

In [ ]:
model_name = "bert-base-multilingual-cased" # Elegimos el modelo BERT multilingüe que vamos a utilizar

tokenizer = AutoTokenizer.from_pretrained(model_name) # Cargamos el tokenizador correspondiente a BERT


# Función que convierte nuestros textos en tokens e IDs
def tokenizar(batch):

    return tokenizer(
        batch["texto_sentimiento"],  # Textos que queremos tokenizar
        truncation=True,             # Si el texto supera el máximo, lo recortamos
        padding="max_length",        # Rellenamos los textos cortos hasta el mismo largo
        max_length=64,               # Todas las secuencias tendrán como máximo 64 tokens

    )


# Aplicamos la función a todo el dataset de entrenamiento (batched=True significa que la función recibe varios ejemplos a la vez)
train_dataset = train_dataset.map(tokenizar, batched=True) # "Recorre el dataset y aplícale la función tokenizar a cada batch"

# Hacemos lo mismo con el dataset de prueba
test_dataset = test_dataset.map(tokenizar, batched=True)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

## 6. Dejando el dataset listo para PyTorch

Faltan dos ajustes antes de que el `Trainer` pueda usar este dataset:

1. El `Trainer` de HuggingFace espera que la columna con la respuesta correcta se llame `labels` (con "s"), no `label`.
2. Le decimos al dataset que devuelva tensores de PyTorch en vez de listas comunes.

In [ ]:
# Cambiamos el nombre de nuestra columna "label"
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

# Indicamos qué columnas necesitamos para entrenar el modelo
columnas_modelo = ["input_ids", "attention_mask", "labels"]

# Convertimos estas columnas al formato de tensores de PyTorch
# para que puedan ser utilizadas directamente por el modelo
train_dataset.set_format("torch", columns=columnas_modelo)
test_dataset.set_format("torch", columns=columnas_modelo)

# Veamos como se ve el primer tweet después de prepararse para entrar en BERT.
train_dataset[0]

{'labels': tensor(0),
 'input_ids': tensor([   101,  11951,  10944,  13595,    108,  56332,  10107,  58768,  66117,
          10165,  12373,  11951,  18163,  23595,  10901,  10189,  10374,  10114,
          15453,  27722,    108,  13375, 101319,  10419,  10111,  11951,  16938,
            112,    188,  13595,  11345,  23595,    102,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}

```
input_ids        → ¿Qué tokens hay?
attention_mask   → ¿Cuáles de esos tokens son reales o padding?
labels           → ¿Cuál es la respuesta correcta?

## 7. Cargando el modelo para el problema binario

Cargamos un modelo nuevo para clasificacion binaria. Indicamos al modelo como se llama cada categoria (`id2label` y `label2id`); no es obligatorio, pero hace que las predicciones se lean mas facil mas adelante.

In [ ]:
id2label = {0: "Negativo", 1: "Positivo"} # mapa que permite convertir los IDs numericos en nombres de categorias
label2id = {"Negativo": 0, "Positivo": 1} # mapa inverso

# Cargamos BERT preparado para una tarea de clasificacion
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # Nuestro problema tiene dos categorias
    id2label=id2label,  # Le decimos al modelo que significa cada ID
    label2id=label2id,  # Le decimos al modelo que ID corresponde a cada categoria
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


>*AutoModelForSequenceClassification* es una clase de Hugging Face que carga un modelo preentrenado (como BERT) y le agrega una capa final para clasificar textos.
>
>En nuestro caso:
>
>BERT + una capa de clasificación → predicción Negativo / Positivo
>
>La palabra Auto significa que Hugging Face detecta automáticamente qué arquitectura debe cargar a partir del nombre del modelo

## 8. El paso critico: congelando las capas base

Esta es la idea central del transfer learning eficiente. BERT tiene millones de parametros que ya aprendieron a "entender" el idioma durante su pre-entrenamiento. Si entrenamos el modelo completo con nuestro dataset (que es chico, comparado con lo que uso BERT originalmente), corremos el riesgo de romper ese conocimiento previo -- esto se conoce como **catastrophic forgetting**.

La solucion: congelamos las capas base del modelo con `requires_grad = False`, de forma que durante el entrenamiento no se actualicen. Solo se va a entrenar la cabeza de clasificacion, que es la parte nueva y que arranca con pesos al azar.

Esta estrategia se llama **feature extraction**: usamos BERT como un generador de representaciones fijo, y entrenamos unicamente un clasificador nuevo encima.

```
TRANSFER LEARNING:

┌───────────────────┐       ┌─────────────────────┐       ┌──────────────────┐
│                   │       │                     │       │                  │
│      📝 TEXTO     │ ────► │   🧠 BERT            │ ────► │  🎯 CLASIFICADOR │
│                   │       │                     │       │                  │
│ "This flight      │       │ MILLONES DE PESOS   │       │  PESOS NUEVOS    │
│  was terrible"    │       │ ya pre-entrenados   │       │  🎲 al azar      │
│                   │       │                     │       │                  │
└───────────────────┘       │ 🔒 CONGELADO        │       │ 🔥 SE ENTRENA    │
                            │                     │       │                  │
                            │ requires_grad=False │       │ Negativo/Positivo│
                            └─────────────────────┘       └──────────────────┘


FEATURE EXTRACTION:

             BERT = generador de representaciones fijo
                              +
                 Clasificador = aprende nuestra tarea



```

In [ ]:
# BERT tiene miliones de parameters ya entrenado
# printamos algunos:

for i, (nombre, param) in enumerate(model.named_parameters()):

    print(i, nombre, param.shape)
    if i == 10:
        break

0 bert.embeddings.word_embeddings.weight torch.Size([119547, 768])
1 bert.embeddings.position_embeddings.weight torch.Size([512, 768])
2 bert.embeddings.token_type_embeddings.weight torch.Size([2, 768])
3 bert.embeddings.LayerNorm.weight torch.Size([768])
4 bert.embeddings.LayerNorm.bias torch.Size([768])
5 bert.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
6 bert.encoder.layer.0.attention.self.query.bias torch.Size([768])
7 bert.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
8 bert.encoder.layer.0.attention.self.key.bias torch.Size([768])
9 bert.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
10 bert.encoder.layer.0.attention.self.value.bias torch.Size([768])


--> Reminder: parameters son todas las variables que se puede aprender (learnable), como los pesos y los sesgos

In [ ]:
# Si queremos ver especificamente los pesos (que vamos a congelar):
pesos = model.bert.embeddings.word_embeddings.weight
print(pesos.flatten()[:50])

tensor([ 2.5951e-02, -6.1734e-03, -4.0997e-03,  3.8244e-02, -3.8685e-02,
         7.6270e-03,  2.3290e-02,  7.2133e-03, -8.2021e-04, -4.6658e-02,
        -3.5500e-02, -1.5062e-02, -2.0823e-02, -5.2826e-02,  6.9264e-03,
         1.3139e-02, -2.2896e-02, -1.9672e-02, -4.9225e-02, -1.7397e-02,
         5.9576e-07, -8.1071e-02, -3.9827e-03,  1.2463e-02, -1.6101e-02,
        -6.2327e-03, -2.3935e-02,  2.5088e-02, -1.8721e-02,  7.3622e-03,
        -2.1641e-02, -3.2469e-02, -5.4654e-02, -2.0303e-02,  3.3343e-02,
         7.5104e-03, -1.8070e-02,  2.6857e-02, -1.3869e-02,  2.9127e-02,
         1.3254e-02,  1.8543e-02,  1.4068e-02,  2.7046e-02,  1.5666e-02,
         7.6299e-03,  1.4827e-02, -3.1548e-03,  4.5813e-03, -1.3269e-02],
       grad_fn=<SliceBackward0>)


In [ ]:
# Congelamos los pesos de las capas base de BERT para que no se actualicen durante el entrenamiento

for param in model.bert.parameters():
    param.requires_grad = False # Desactivamos el calculo de gradientes para todos los parametros que existe en BERT

Verifiquemos que el congelado funciono, contando cuantos parametros del modelo son entrenables ahora comparado con el total.

In [ ]:
# Funcion para contar los parametros del modelo
def contar_parametros(modelo):

    # Contamos TODOS los parametros del modelo (numel() devuelve la cantidad de valores dentro de cada tensor)
    total = sum(p.numel() for p in modelo.parameters())

    # Contamos solamente los parametros que se pueden actualizar (requires_grad=True significa que participan en el aprendizaje)
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, entrenables


total, entrenables = contar_parametros(model)
print(f"Parametros totales: {total:,}")
print(f"Parametros entrenables: {entrenables:,}")
print(f"Porcentaje entrenable: {100 * entrenables / total:.5f}%")

Parametros totales: 177,854,978
Parametros entrenables: 1,538
Porcentaje entrenable: 0.00086%


Deberias ver que el porcentaje entrenable es muy chico (menos del 1%): son los pesos de la cabeza de clasificacion, la unica parte que va a cambiar durante el entrenamiento. Los mas de 100 millones de parametros de BERT se quedan tal cual estan, protegiendo el conocimiento que ya tienen.

## 9. Bonus: preparando tambien la version con tres categorias

El temario original pide un minimo de 2 categorias, y eso es lo que vamos a entrenar la clase que viene. Pero quizas te preguntes: ¿que cambiaria si usaramos las tres categorias originales (positive, neutral, negative)? Preparemos esa version tambien, para comparar. Fijate que el codigo es practicamente el mismo, solo cambia el mapeo de etiquetas y el numero de categorias del modelo.

In [ ]:
etiqueta2id_tres = {"negative": 0, "neutral": 1, "positive": 2}

df_tres = df.copy()
df_tres["label"] = df_tres["sentimiento_original"].map(etiqueta2id_tres)

df_muestra_tres = df_tres.sample(n=3000, random_state=42)

train_df_tres, test_df_tres = train_test_split(
    df_muestra_tres,
    test_size=0.2,
    stratify=df_muestra_tres["label"],
    random_state=42,
)

train_dataset_tres = Dataset.from_pandas(
    train_df_tres[["texto_sentimiento", "label"]].reset_index(drop=True)
)
test_dataset_tres = Dataset.from_pandas(
    test_df_tres[["texto_sentimiento", "label"]].reset_index(drop=True)
)

train_dataset_tres = train_dataset_tres.map(tokenizar, batched=True)
test_dataset_tres = test_dataset_tres.map(tokenizar, batched=True)

train_dataset_tres = train_dataset_tres.rename_column("label", "labels")
test_dataset_tres = test_dataset_tres.rename_column("label", "labels")
train_dataset_tres.set_format("torch", columns=columnas_modelo)
test_dataset_tres.set_format("torch", columns=columnas_modelo)

In [ ]:
id2label_tres = {0: "Negativo", 1: "Neutral", 2: "Positivo"}
label2id_tres = {"Negativo": 0, "Neutral": 1, "Positivo": 2}

model_tres = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label_tres,
    label2id=label2id_tres,
)

for param in model_tres.bert.parameters():
    param.requires_grad = False

total_tres, entrenables_tres = contar_parametros(model_tres)
print(f"Version binaria     -> entrenables: {entrenables:,}")
print(f"Version tres clases -> entrenables: {entrenables_tres:,}")

Vas a ver que el numero de parametros entrenables es un poquito mas alto en la version de tres categorias: la cabeza de clasificacion tiene una neurona de salida mas, asi que tiene un poco mas de pesos para entrenar. La cantidad de parametros congelados (los de BERT) es exactamente igual en los dos casos.

Todavia no entrenamos ninguna de las dos versiones -- eso es lo que hacemos en el proximo notebook. Pero ya dejamos la version de tres categorias preparada para poder comparar los resultados finales.

## Resumen

- Simplificamos el dataset a un problema binario (Positivo / Negativo) para el ejercicio principal, dejando la version de tres categorias preparada como comparacion.
- Separamos los datos en train y test con `train_test_split`, usando `stratify` para mantener la proporcion de clases.
- Tokenizamos el dataset completo en batch con `.map()`, usando `padding` y `truncation` para que todos los ejemplos tengan el mismo largo.
- Dejamos el dataset en el formato que espera el `Trainer`: columna `labels` y tensores de PyTorch.
- Aplicamos el paso central de Fase 3: congelamos las capas base de BERT con `requires_grad = False`, y verificamos que menos del 1% de los parametros del modelo quedaron entrenables.

En el proximo notebook completamos la Fase 4: configuramos el `Trainer`, entrenamos el modelo y probamos como funciona con una frase nueva.